# Schrödinger Deluxe Ultra - Eredmény elemző
Last update: _2026.05.10._

In [21]:
# ide kell beilleszteni a gulyakor eredményeoldalának URL-jét
GULYAKOR_URL = "https://gulyakor.hu/run/2748"

## Adatbetöltés

In [22]:
import pandas as pd

In [23]:
pdf_data = pd.read_html(GULYAKOR_URL)
pdf_data = pdf_data[0].copy()
pdf_data.drop([0, 1, 2, 3], inplace=True)

In [24]:
pdf_data['Leolvasás ideje'] = pd.to_datetime(pdf_data['Leolvasás ideje'], format='%Y-%m-%d %H:%M:%S')

In [25]:
# Szakaszidő parse többféle formátumra (pl. HH:MM:SS, MM:SS, '1h 2m 3s', stb.)
sz_raw = pdf_data['Szakaszidő'].astype(str).str.strip()

# 1) Első próbálkozás: általános timedelta parser
sz_td = pd.to_timedelta(sz_raw, errors='coerce')

# 2) Fallback: ha valaki MM:SS formában van, alakítsuk HH:MM:SS-re
mask = sz_td.isna() & sz_raw.str.match(r'^\d{1,2}:\d{2}$', na=False)
if mask.any():
    sz_td.loc[mask] = pd.to_timedelta('00:' + sz_raw.loc[mask], errors='coerce')

pdf_data['Szakaszidő'] = sz_td

In [26]:
clean_meter = lambda x: x if pd.isna(x) else str(x).replace('\xa0', '').replace(' ', '').replace(',', '.').replace('méter', '').replace('meter', '').rstrip('m').strip()
for col in ['Szintemelkedés', 'Szintesés', 'Szakaszhossz', 'Megtett távolság']:
    pdf_data[col] = pd.to_numeric(pdf_data[col].apply(clean_meter), errors='coerce')

In [27]:
pdf_filtered = pdf_data[
    pdf_data['Szakaszidő'].notna() &
    (pdf_data['Szakaszhossz'] > 0)
].copy()

In [28]:
print("Szűrt adatok száma: {:<5d}".format(len(pdf_filtered)))

Szűrt adatok száma: 35   


In [29]:
pdf_filtered['Távolság'] = pdf_filtered['Megtett távolság'].ffill().diff().fillna(pdf_filtered['Megtett távolság'].ffill())


In [30]:
# Speed in km/h from distance (m) and section time
pdf_filtered['Sebesség'] = (pdf_filtered['Távolság'] / 1000) / (pdf_filtered['Szakaszidő'].dt.total_seconds() / 3600)

In [31]:
pdf_filtered[:20]

,Ellenőrzőpont,Futó,Leolvasás ideje,Szakaszidő,Teljes idő,Szintemelkedés,Szintesés,Szakaszhossz,Megtett távolság,Távolság,Sebesség
5,Harangvirág út,LÉP CSABA,2026-05-09 08:34:13,0 days 00:32:41,34:13,120.0,240.0,5290.0,5290.0,5290.0,9.711372
6,Északkeleti átjáró,LÉP CSABA,2026-05-09 09:29:05,0 days 00:54:52,01:29:05,310.0,200.0,7880.0,13170.0,7880.0,8.617254
8,Bányaudvar,Balogh Zoltán,2026-05-09 10:41:12,0 days 01:01:26,02:41:12,190.0,240.0,8210.0,21380.0,8210.0,8.018448
9,Megyehatár,Balogh Zoltán,2026-05-09 11:07:40,0 days 00:26:28,03:07:40,90.0,80.0,3070.0,24450.0,3070.0,6.959698
11,Pihenő,Banra H,2026-05-09 11:24:11,0 days 00:16:24,03:24:11,140.0,30.0,1760.0,26210.0,1760.0,6.439024
12,Homok-nyereg,Banra H,2026-05-09 11:36:41,0 days 00:12:30,03:36:41,20.0,90.0,1710.0,27920.0,1710.0,8.208000
13,Száz éves fenyő,Banra H,2026-05-09 11:44:09,0 days 00:07:28,03:44:09,20.0,50.0,1160.0,29080.0,1160.0,9.321429
14,Fenyősor teteje,Banra H,2026-05-09 11:53:29,0 days 00:09:20,03:53:29,30.0,40.0,860.0,29940.0,860.0,5.528571
15,Tinnye-hegy,Banra H,2026-05-09 12:18:25,0 days 00:24:56,04:18:25,110.0,70.0,3470.0,33410.0,3470.0,8.350267
17,Sas-szikla,Rádi Gábor,2026-05-09 13:37:58,0 days 00:59:45,05:37:58,170.0,200.0,6070.0,39480.0,6070.0,6.095397


## Elemzés

### Versenyzők átlagsebessége

In [32]:
# write code here
# Average speed/pace + total incline/decline per runner
runner_speed_pace_df = (
    pdf_filtered.dropna(subset=['Futó', 'Távolság', 'Szakaszidő'])
    .assign(hours=lambda d: d['Szakaszidő'].dt.total_seconds() / 3600)
    .groupby('Futó', as_index=False)
    .agg(
        total_distance_m=('Távolság', 'sum'),
        total_hours=('hours', 'sum'),
        total_incline_m=('Szintemelkedés', 'sum'),
        total_decline_m=('Szintesés', 'sum')
    )
)
runner_speed_pace_df['avg_speed_kmh'] = (runner_speed_pace_df['total_distance_m'] / 1000) / runner_speed_pace_df['total_hours']
runner_speed_pace_df['avg_pace_min_per_km'] = 60 / runner_speed_pace_df['avg_speed_kmh']
runner_speed_pace_df = runner_speed_pace_df.sort_values('avg_speed_kmh', ascending=False).reset_index(drop=True)
runner_speed_pace_df


,Futó,total_distance_m,total_hours,total_incline_m,total_decline_m,avg_speed_kmh,avg_pace_min_per_km
0,LÉP CSABA,28470.0,3.557222,1050.0,900.0,8.003436,7.496780
1,Antal Zoltán,24000.0,3.360833,710.0,650.0,7.141086,8.402083
2,Balogh Zoltán,20460.0,2.897500,550.0,630.0,7.061260,8.497067
3,Biro Balazs,41900.0,6.150556,1070.0,1030.0,6.812393,8.807478
4,Banra H,22490.0,3.321667,630.0,840.0,6.770697,8.861716
5,Rádi Gábor,9960.0,1.802778,340.0,300.0,5.524807,10.860107


### Legnagyobb szintkülönbség

In [33]:
# Aggregate by runner: incline, decline, and distance ran
runner_agg = pdf_data.groupby('Futó', as_index=False).agg(
    total_incline_m=('Szintemelkedés', 'sum'),
    total_decline_m=('Szintesés', 'sum'),
    total_ran_m=('Szakaszhossz', 'sum')
)
runner_agg['total_ran_km'] = runner_agg['total_ran_m'] / 1000

most_incline = runner_agg.loc[runner_agg['total_incline_m'].idxmax()]
least_incline = runner_agg.loc[runner_agg['total_incline_m'].idxmin()]
most_decline = runner_agg.loc[runner_agg['total_decline_m'].idxmax()]
least_decline = runner_agg.loc[runner_agg['total_decline_m'].idxmin()]
most_ran = runner_agg.loc[runner_agg['total_ran_m'].idxmax()]
least_ran = runner_agg.loc[runner_agg['total_ran_m'].idxmin()]

print(f"Most incline: {most_incline['Futó']} ({most_incline['total_incline_m']:.0f} m)")
print(f"Least incline: {least_incline['Futó']} ({least_incline['total_incline_m']:.0f} m)")
print(f"Most decline: {most_decline['Futó']} ({most_decline['total_decline_m']:.0f} m)")
print(f"Least decline: {least_decline['Futó']} ({least_decline['total_decline_m']:.0f} m)")
print(f"Most ran: {most_ran['Futó']} ({most_ran['total_ran_km']:.2f} km)")
print(f"Least ran: {least_ran['Futó']} ({least_ran['total_ran_km']:.2f} km)")

runner_agg.sort_values('total_ran_m', ascending=False)


Most incline: Biro Balazs (1070 m)
Least incline: Rádi Gábor (340 m)
Most decline: Biro Balazs (1030 m)
Least decline: Rádi Gábor (300 m)
Most ran: Biro Balazs (41.90 km)
Least ran: Rádi Gábor (9.96 km)


,Futó,total_incline_m,total_decline_m,total_ran_m,total_ran_km
3,Biro Balazs,1070.0,1030.0,41900.0,41.90
4,LÉP CSABA,1050.0,900.0,28470.0,28.47
0,Antal Zoltán,710.0,650.0,24000.0,24.00
2,Banra H,630.0,840.0,22490.0,22.49
1,Balogh Zoltán,550.0,630.0,20460.0,20.46
5,Rádi Gábor,340.0,300.0,9960.0,9.96


### Hatékonysági pontszám: távolság függőleges méterenként

In [34]:
# 1) Efficiency score: distance per vertical meter
runner_totals_eff = pdf_filtered.groupby('Futó', as_index=False).agg(
    total_incline_m=('Szintemelkedés', 'sum'),
    total_decline_m=('Szintesés', 'sum'),
    total_distance_m=('Távolság', 'sum')
)
runner_totals_eff['total_distance_km'] = runner_totals_eff['total_distance_m'] / 1000
eff = runner_totals_eff.copy()
eff['vertical_m'] = eff['total_incline_m'] + eff['total_decline_m']
eff = eff[eff['vertical_m'] > 0].copy()
eff['eff_km_per_vertical_km'] = eff['total_distance_km'] / (eff['vertical_m'] / 1000)
eff.sort_values('eff_km_per_vertical_km', ascending=False)


,Futó,total_incline_m,total_decline_m,total_distance_m,total_distance_km,vertical_m,eff_km_per_vertical_km
3,Biro Balazs,1070.0,1030.0,41900.0,41.90,2100.0,19.952381
0,Antal Zoltán,710.0,650.0,24000.0,24.00,1360.0,17.647059
1,Balogh Zoltán,550.0,630.0,20460.0,20.46,1180.0,17.338983
5,Rádi Gábor,340.0,300.0,9960.0,9.96,640.0,15.562500
2,Banra H,630.0,840.0,22490.0,22.49,1470.0,15.299320
4,LÉP CSABA,1050.0,900.0,28470.0,28.47,1950.0,14.600000


- `vertical_m`: a teljes függőleges mozgás méterben, a `total_incline_m + total_decline_m` képlettel számítva.
- `eff_km_per_vertical_km`: hány vízszintes kilométert tesz meg minden egyes függőleges kilométer, a `total_distance_km / (vertical_m / 1000)` képlettel számítva.

A magasabb `eff_km_per_vertical_km` érték azt jelenti, hogy az útvonal viszonylag laposabb (nagyobb távolság egységnyi emelkedés + lejtő), míg az alacsonyabb értékek dombosabb útvonalakat jeleznek.

### Tempóeltolódás (pace drift): késői vs. korai tempó

In [35]:
# 2) Pace drift: late-vs-early pace (positive = slowed down)
pace_source = (
    pdf_filtered.dropna(subset=['Futó', 'Sebesség'])
    .loc[lambda d: d['Sebesség'] > 0]
    .copy()
)
pace_source['pace_min_per_km'] = 60 / pace_source['Sebesség']

drift_rows = []
for runner, g in pace_source.groupby('Futó'):
    g = g.sort_index()
    n = len(g)
    if n < 2:
        continue
    half = n // 2
    early = g.iloc[:half]
    late = g.iloc[half:]
    early_pace = early['pace_min_per_km'].mean()
    late_pace = late['pace_min_per_km'].mean()
    drift_rows.append({
        'Futó': runner,
        'early_pace_min_per_km': early_pace,
        'late_pace_min_per_km': late_pace,
        'pace_drift_min_per_km': late_pace - early_pace
    })

pace_drift = pd.DataFrame(drift_rows).sort_values('pace_drift_min_per_km')

In [36]:
pace_drift

,Futó,early_pace_min_per_km,late_pace_min_per_km,pace_drift_min_per_km
0,Antal Zoltán,8.001567,8.585481,0.583913
2,Banra H,8.479405,9.170850,0.691445
1,Balogh Zoltán,8.051904,9.103829,1.051925
4,LÉP CSABA,7.196312,8.365162,1.168850
3,Biro Balazs,7.755765,10.151490,2.395725
5,Rádi Gábor,9.843493,12.673936,2.830443


- `early_pace_min_per_km`: az első félidő átlagtempója perc/km-ben.
- `late_pace_min_per_km`: a második félidő átlagtempója perc/km-ben.
- `pace_drift_min_per_km`: a tempóváltozás (`late - early`) perc/km-ben.

Értelmezés: pozitív `pace_drift_min_per_km` esetén a futó lassult a második felére, negatív értéknél gyorsult, 0 körüli értéknél pedig közel egyenletes tempót futott.

### Sebesség emelkedőn vs. lejtőn

In [37]:
# 3) Climb-heavy vs descent-heavy speed
terrain = pdf_filtered.dropna(subset=['Futó', 'Szintemelkedés', 'Szintesés', 'Sebesség']).copy()
terrain['terrain_type'] = 'flat/mixed'
terrain.loc[terrain['Szintemelkedés'] > terrain['Szintesés'], 'terrain_type'] = 'climb-heavy'
terrain.loc[terrain['Szintesés'] > terrain['Szintemelkedés'], 'terrain_type'] = 'descent-heavy'
terrain_speed = terrain.groupby(['Futó', 'terrain_type'], as_index=False)['Sebesség'].mean()
terrain_speed['Pace_min_per_km'] = 60 / terrain_speed['Sebesség']
terrain_speed


,Futó,terrain_type,Sebesség,Pace_min_per_km
0,Antal Zoltán,climb-heavy,6.751017,8.887550
1,Antal Zoltán,descent-heavy,8.360277,7.176796
2,Balogh Zoltán,climb-heavy,6.959698,8.621064
3,Balogh Zoltán,descent-heavy,7.560679,7.935795
4,Balogh Zoltán,flat/mixed,6.147279,9.760417
5,Banra H,climb-heavy,7.394646,8.113979
6,Banra H,descent-heavy,7.114980,8.432911
7,Biro Balazs,climb-heavy,6.547284,9.164106
8,Biro Balazs,descent-heavy,7.415552,8.091103
9,LÉP CSABA,climb-heavy,7.523074,7.975463


In [38]:
print("Átlagos sebesség tereptípusonként")
print("- Emelkedő: {:>6.2f} km/h".format(
    terrain_speed[terrain_speed['terrain_type'] == 'climb-heavy'].sort_values('Sebesség', ascending=False)['Sebesség'].mean()
))
print("- Lejtő: {:>9.2f} km/h".format(terrain_speed[terrain_speed['terrain_type'] == 'descent-heavy'].sort_values('Sebesség', ascending=False)['Sebesség'].mean()))
print("- Flat/mixed: {:>4.2f} km/h".format(terrain_speed[terrain_speed['terrain_type'] == 'flat/mixed'].sort_values('Sebesség', ascending=False)['Sebesség'].mean()))

Átlagos sebesség tereptípusonként
- Emelkedő:   6.74 km/h
- Lejtő:      7.28 km/h
- Flat/mixed: 6.15 km/h


`terrain_speed` értelmezés:

A futók átlagsebessége tereptípus szerint jól elkülönül: a `descent-heavy` szakaszokon a legmagasabb (átlag ~7.28 km/h), ezt követi a `climb-heavy` (~6.74 km/h), míg a `flat/mixed` a legalacsonyabb (~6.15 km/h).

A `descent-heavy` és `climb-heavy` összevetésben a lejtős szakaszok átlagosan kb. +0.55 km/h előnyt mutatnak. A 6 futóból 4-nél gyorsabb a lejtős profil, tehát az összkép alapján a lejtés általában tempónövelő hatású, de egyéni eltérések továbbra is láthatók.


In [39]:
print("Átlagos tempó tereptípusonként")
print("- Emelkedő: {:>6.2f} min/km".format(
    terrain_speed[terrain_speed['terrain_type'] == 'climb-heavy'].sort_values('Pace_min_per_km', ascending=False)['Pace_min_per_km'].mean()
))
print("- Lejtő: {:>9.2f} min/km".format(terrain_speed[terrain_speed['terrain_type'] == 'descent-heavy'].sort_values('Pace_min_per_km', ascending=False)['Pace_min_per_km'].mean()))
print("- Flat/mixed: {:>4.2f} min/km".format(terrain_speed[terrain_speed['terrain_type'] == 'flat/mixed'].sort_values('Pace_min_per_km', ascending=False)['Pace_min_per_km'].mean()))

Átlagos tempó tereptípusonként
- Emelkedő:   9.03 min/km
- Lejtő:      8.44 min/km
- Flat/mixed: 9.76 min/km


### Konzisztencia index: az alacsonyabb standard tempó egyenletesebb tempót jelent

In [42]:
# 5) Consistency index: lower std pace means steadier pacing
consistency_source = (
    pdf_filtered.dropna(subset=['Futó', 'Sebesség'])
    .loc[lambda d: d['Sebesség'] > 0]
    .copy()
)
consistency_source['pace_min_per_km'] = 60 / consistency_source['Sebesség']

consistency = consistency_source.groupby('Futó', as_index=False).agg(
    mean_pace_min_per_km=('pace_min_per_km', 'mean'),
    std_pace_min_per_km=('pace_min_per_km', 'std'),
    section_count=('pace_min_per_km', 'count')
)
consistency.sort_values('std_pace_min_per_km')

,Futó,mean_pace_min_per_km,std_pace_min_per_km,section_count
1,Balogh Zoltán,8.577867,0.933925,4
4,LÉP CSABA,7.864226,1.032222,7
0,Antal Zoltán,8.390843,1.448463,3
3,Biro Balazs,8.953627,2.013986,10
5,Rádi Gábor,11.730455,2.055233,3
2,Banra H,8.825127,2.379540,8


A `consistency` tábla futónként mutatja az átlagtempót (`mean_pace_min_per_km`) és a tempó szórását (`std_pace_min_per_km`).

- Minél kisebb a `std_pace_min_per_km`, annál egyenletesebb volt a tempó.
- Minél nagyobb a `std_pace_min_per_km`, annál jobban ingadozott a tempó a szakaszok között.
- Az értelmezésnél figyeljünk a `section_count` oszlopra is: kevés szakasz esetén a szórás kevésbé stabil becslés.

Gyakorlatban érdemes a futókat elsősorban `std_pace_min_per_km` szerint rendezni, majd a hasonló szórású eredményeket `mean_pace_min_per_km` alapján összehasonlítani.